### Function approximation of a cost-to-go

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro860/pendulum_cost_to_go_approximation.ipynb)

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox.


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink import (
    DynamicProgrammingPlanner,
    InvertedPendulum,
    LQRPlanner,
    PlanningProblem,
    QuadraticCost,
    StateSpaceGrid,
)

**Defining a dynamic system model**

Here we load a already defined class including all the dynamic equations and we define the domain (for states x and torque u) for which we will generate a controller.

In [ ]:
sys = InvertedPendulum()

# State and control input domain
sys.inputs["u"].upper_bound = np.array([+5.0])  # Max torque
sys.inputs["u"].lower_bound = np.array([-5.0])  # Min torque

sys.state.upper_bound = np.array([+6.0, +6.0])  # Max angle, max velocity
sys.state.lower_bound = np.array([-6.0, -6.0])  # Min angle, min velocity

# sys.state.upper_bound = np.array([+0.1, +0.1])
# sys.state.lower_bound = np.array([-0.1, -0.1])

**Defining the cost function**

Here we can define a cost function or the type:

$J = \int_{0}^{t_f}  g( x , u , t ) dt + h(x_f,t_f)$

In [ ]:
# Cost Function
qcf = QuadraticCost.from_system(sys, xbar=np.array([0.0, 0.0]))  # target
INF = 200.0

Here we define the parameters used by the cost function

**Synthetizing the "optimal" controller**

*VI controller*

Here we use a library function that: \\
1) Discretize the domain of the state and control inputs, by default the 2D state-space is discretized into a 101 x 101 grid, the torque is dicretized into 11 discrete level, and the time step is 0.05 sec. \\
2) Use the value-iteraton (default is 300 iterations) to compute optimal cost to go and control actions based on the previously defined cost function g(x,u,t) \\
3) Generate a continuous control law by interpolating in the computed discrete solution

In [ ]:
x_grid = (201, 201)
u_grid = (21,)
dt = 0.05

problem = PlanningProblem(sys, cost=qcf, X=sys.state.box, infeasible_cost=INF)
grid_sys = StateSpaceGrid(problem, x_grid_shape=x_grid, u_grid_shape=u_grid, dt=dt)

Here we use the algorithm called value iteration to solve for an (approximate) solution to the bellman equations.

In [ ]:
dp = DynamicProgrammingPlanner(problem, grid=grid_sys, tol=0.1)
dp.solve()

The following figure illustrate the computed optimal cost-to-go J* for every starting state. Note that we saturate the maximum J* to plot here to better show the range of interest.

In [ ]:
dp.clean_infeasible_set()

max_j_to_plot = 300

dp.plot_cost2go(jmax=max_j_to_plot)

In [ ]:
dp.plot_cost2go(jmax=max_j_to_plot, show_3d=True)

# Function approximation

## Approximator classes

In [ ]:
###############################################################################
### Linear Function approximator
###############################################################################

class LinearFunctionApproximator():

    ############################
    def __init__(self, n = 10 ):

        self.n = n # number of parameters


    ############################
    def J_hat( self , x , w ):
        """ Compute J approx given state x and param w """

        phi   = self.compute_kernel( x )

        J_hat = phi.T @ w

        return J_hat


    ############################
    def dJ_dw( self , x ):
        """ Compute dJ/dw given state x """

        phi   = self.compute_kernel( x )

        return phi


    ############################
    def compute_kernel( self , x ):
        """ Compute kernel functions phi given a state x """

        phi = np.zeros( self.n )

        return phi


    ############################
    def compute_all_kernel( self , Xs ):
        """ Compute all kernel functions phi given a m state x """

        m = Xs.shape[0]   # number of data point
        n = self.n        # number of params

        P = np.zeros( ( m , n ))

        for i in range(m):

            P[i,:] = self.compute_kernel( Xs[i,:] )

        return P.T


    ############################
    def least_square_fit( self , Js , P ):
        """ solve J_d = P w """

        #P = self.compute_all_kernel( Xs )

        w = np.linalg.lstsq( P.T , Js , rcond=None)[0]

        J_hat = P.T @ w

        return w , J_hat



###############################################################################
class MultipleGaussianFunctionApproximator( LinearFunctionApproximator ):

    ############################
    def __init__(self, Xs , sig = 1.0 ):
        """
        J_hat = sum exp( - || x - x0 || / 2 sig^2 )

        """

        self.Xs    = Xs
        self.sys_n = Xs.shape[1]
        self.n     = Xs.shape[0]   # number of data point
        self.a     = -0.5  / (sig**2)


    ############################
    def compute_kernel( self , x ):
        """ return approx a state x """

        phi = np.zeros(self.n)

        for i in range(self.n):

            e = x - self.Xs[i,:]
            r = e.T @ e

            phi[i] = np.exp( self.a * r )

        return phi


###############################################################################
### Quadratic Function approximator
###############################################################################

class QuadraticFunctionApproximator( LinearFunctionApproximator ):

    ############################
    def __init__(self, sys_n = 2 , x0 = None):
        """
        J_hat = C + B x + x' A x = w' phi

        """

        self.sys_n = sys_n

        if x0 is not None:

            self.x0 = x0

        else:

            self.x0 = np.zeros( sys_n )

        self.n_2_diag = sys_n
        self.n_2_off  = int((sys_n**2-sys_n)/2)
        self.n_2      = +self.n_2_diag + self.n_2_off # 2nd order number of weight
        self.n_1      = sys_n                    # 1nd order number of weight
        self.n_0      = 1                        # 0nd order number ofweight

        # Total number of parameters
        self.n = int(self.n_2 + self.n_1 + self.n_0)


    ############################
    def compute_kernel( self , x ):
        """ return approx a state x """

        phi = np.zeros( self.n )

        x = x - self.x0

        xxT = np.outer( x , x )

        #indices
        n0 = self.n_0
        n1 = self.n_0 + self.n_1
        n2 = self.n_0 + self.n_1 + self.n_2_diag
        n3 = self.n_0 + self.n_1 + self.n_2_diag + self.n_2_off

        phi[0]     = 1
        phi[n0:n1] = x
        phi[n1:n2] = np.diag( xxT )
        phi[n2:n3] = xxT[np.triu_indices( self.sys_n, k = 1)]

        return phi

# Gaussian approx

Creating a ( res x res ) grid of gaussian

In [ ]:
res = 20

grid_sys_gaussian = StateSpaceGrid(
    problem, x_grid_shape=(res, res), u_grid_shape=(3,), dt=0.05, precompute=False
)
approx = MultipleGaussianFunctionApproximator(grid_sys_gaussian.states)

## Supervised learning


In [ ]:
J = dp.result.J  # cost-to-go at every node of grid_sys

phi = approx.compute_all_kernel(grid_sys.states)  # Phi for all point on the grid

w, J_hat = approx.least_square_fit(J, phi)

In [ ]:
grid_sys.plot_value(J_hat, show_3d=True, title="Gaussian approx")

fig, ax = grid_sys.plot_value(J, show_3d=True, title="J vs. J_hat", show=False)
Z2 = grid_sys.slice_2d(grid_sys.grid_from_array(J_hat), 0, 1)
X, Y = np.meshgrid(grid_sys.x_levels[0], grid_sys.x_levels[1])
ax.plot_wireframe(X, Y, Z2.T)
plt.show()

# Quadratic Approx

In [ ]:
qfa = QuadraticFunctionApproximator(sys.n, x0=qcf.xbar)

Xs = grid_sys.states  # All state on the grid

P = qfa.compute_all_kernel(Xs)

w, J_hat = qfa.least_square_fit(J, P)

grid_sys.plot_value(J_hat, show_3d=True, title="Quadratic approx")

fig, ax = grid_sys.plot_value(J, show_3d=True, title="J vs. J_hat", show=False)
Z2 = grid_sys.slice_2d(grid_sys.grid_from_array(J_hat), 0, 1)
X, Y = np.meshgrid(grid_sys.x_levels[0], grid_sys.x_levels[1])
ax.plot_wireframe(X, Y, Z2.T)
plt.show()

# Linarized LQR Solution

In [ ]:
lqr = LQRPlanner(problem).solve()  # the Riccati solution of the same quadratic problem, linearized at xbar
S = lqr.solver.P

w_riccati = np.array([0, 0, 0, S[0, 0], S[1, 1], S[0, 1]])
J_riccati = P.T @ w_riccati

grid_sys.plot_value(J_riccati, show_3d=True, title="J_riccati")

fig, ax = grid_sys.plot_value(J, show_3d=True, title="J_riccati vs. J_hat", show=False)
Z2 = grid_sys.slice_2d(grid_sys.grid_from_array(J_riccati), 0, 1)
X, Y = np.meshgrid(grid_sys.x_levels[0], grid_sys.x_levels[1])
ax.plot_wireframe(X, Y, Z2.T)
plt.show()